In [2]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# load data
df = pd.read_csv("C:\\Users\\Dan\\Downloads\\speeches_sentiment_5000.csv").copy()

# sort by time
df["speech_begin"] = pd.to_datetime(df["speech_begin"], errors="coerce")
df = df.sort_values("speech_begin").copy()

# optional: shorten text for CPU training
df["speech_text"] = df["speech_text"].fillna("").str.slice(0, 500)

# train / val / test split
train_end = int(len(df) * 0.70)
val_end = int(len(df) * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

# keep only needed columns
train_df = train_df[["speech_text", "motion_passed"]].rename(columns={"motion_passed": "label"})
val_df = val_df[["speech_text", "motion_passed"]].rename(columns={"motion_passed": "label"})
test_df = test_df[["speech_text", "motion_passed"]].rename(columns={"motion_passed": "label"})

# model name
MODEL_NAME = "GroNLP/bert-base-dutch-cased"

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["speech_text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

# convert to Hugging Face datasets
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

# remove unnecessary columns
train_ds = train_ds.remove_columns(["speech_text"])
val_ds = val_ds.remove_columns(["speech_text"])
test_ds = test_ds.remove_columns(["speech_text"])

# model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "roc_auc": roc_auc_score(labels, probs)
    }

training_args = TrainingArguments(
    output_dir="./bert_motion_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# train
trainer.train()

# evaluate
test_metrics = trainer.evaluate(test_ds)
print("=== BERT ===")
print(test_metrics)

c:\Users\Dan\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dan\.cache\huggingface\hub\models--GroNLP--bert-base-dutch-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 750/750 [00:00<00:00, 7847.62 examples/s]
Some weights of BertForSe

Epoch,Training Loss,Validation Loss,Accuracy,F1,Roc Auc
1,0.709900,0.705154,0.473333,0.100228,0.506973
2,0.681100,0.712808,0.500000,0.432678,0.501328


c:\Users\Dan\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\Dan\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


=== BERT ===
{'eval_loss': 0.7058468461036682, 'eval_accuracy': 0.43066666666666664, 'eval_f1': 0.19887429643527205, 'eval_roc_auc': 0.5449122468373225, 'eval_runtime': 67.9247, 'eval_samples_per_second': 11.042, 'eval_steps_per_second': 2.768, 'epoch': 2.0}
